In [469]:
import sys 
sys.path.append(r"C:\Users\g0004878\Desktop\My codes\Frequently used codes\Frequently-used-codes")
sys.path.append(r"C:\Users\g0004878\Desktop\Projects in FY25-26\utils_files")
import Snowflake_configuration
snowflake_conn_prop = Snowflake_configuration.ds1_role_json
from snowflake.snowpark.session import Session
import pandas as pd
pd.set_option('display.max_columns',None)

### Activate the session

In [470]:
session = Session.builder.configs(snowflake_conn_prop).create()

### Import libraries

In [471]:
import snowflake.snowpark as snowpark
from snowflake.snowpark import functions as F
from snowflake.snowpark.window import Window

### Configuration

In [472]:
TFT_FORECAST_TABLE      = 'MOP_DATABASE.SOQ.PREDICTIONS_BY_TFT_JAN_26_TO_APR_26'
OUTPUT_TABLE            = 'MOP_DATABASE.SOQ.TFT_PREDICTIONS_DISAGGREGATED_SKU_LEVEL'
ECR_TABLE               = 'ANALYTICS_DATABASE.ANALYTICS_SALES.CUSTOMER_RETAILS'
SKU_SUPERCEDENCE_TABLE  = 'MOP_DATABASE.SOQ.SKU_SUPERCEDENCE_MODEL_FAMILY_FEB_2025_UPDATED_V2'
OBD_MAPPING_TABLE       = 'MOP_DATABASE.SOQ.OBD2_MAPPING_VIEW'
PARENT_DEALER_VIEW      = 'FIVETRAN_DATABASE.ORACLE_LDP_OLAP_SCHEMA.WC_INT_ORG_DH'
CUSTOMER_TYPE           = ('Individual',)
IS_OBD                  = True

In [473]:
def get_shape(df):
    return (df.count(), len(df.columns))

In [474]:
def get_null_counts(df):
    null_exprs = [
        F.count(F.iff(F.col(c).is_null(), F.lit(1), F.lit(None))).alias(f"NUMBER_OF_NULL_VALUES_IN_{c}")
        for c in df.columns
    ]

### Step 1

In [475]:
forecast = session.table(TFT_FORECAST_TABLE)

In [476]:
#Data quality check
forecast.group_by(F.col("PARENT_DEALER_CODE_MODEL_FAMILY"),F.col("MONTH_OF_SALE")).agg(F.count_distinct(F.col("PREDICTED_SALES_TFT")).alias("UNIQUE_COUNT_OF_NET_SALES")).order_by(F.col("UNIQUE_COUNT_OF_NET_SALES").desc()).show()

-------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"              |"MONTH_OF_SALE"  |"UNIQUE_COUNT_OF_NET_SALES"  |
-------------------------------------------------------------------------------------------------
|12075<>SPLENDOR+<>DRUM<>SELF<>CAST<>GREY       |2026-03-01       |1                            |
|10879<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK      |2026-03-01       |1                            |
|11576<>PLEASURE+<>DRUM<>SELF<>CAST<>RED        |2026-02-01       |1                            |
|11002<>HF DELUXE<>DRUM<>SELF<>CAST<>RED BLACK  |2026-01-01       |1                            |
|10118<>HF DELUXE<>DRUM<>SELF<>CASTI<>BLACK     |2026-02-01       |1                            |
|11576<>PLEASURE+<>DRUM<>SELF<>CAST<>RED        |2026-01-01       |1                            |
|11981<>PASSION<>DRUM<>SELF<>CAST<>BLACK        |2026-04-01       |1                            |
|12015<>HF DELUXE<>D

In [477]:
#Convert MONTH_OF_SALE to Date type
forecast = forecast.with_column("MONTH_OF_SALE",F.to_date(F.col("MONTH_OF_SALE")))

#Fetching the dealer code
forecast = forecast.with_column("PARENT_DEALER_CODE",
                                F.trim(F.split(F.col("PARENT_DEALER_CODE_MODEL_FAMILY"),
                                F.lit('<>'))[0]))

#Fetching the part after the dealer code
forecast = forecast.with_column("UNIQUE_FAMILY_CODE",F.substr(F.col("PARENT_DEALER_CODE_MODEL_FAMILY"),
                                (F.charindex(F.lit('<>'),F.col("PARENT_DEALER_CODE_MODEL_FAMILY")))+F.lit(2)
                                ))

In [478]:
forecast.columns

['PARENT_DEALER_CODE_MODEL_FAMILY',
 'PREDICTED_SALES_TFT',
 'MONTH_OF_SALE',
 'PARENT_DEALER_CODE',
 'UNIQUE_FAMILY_CODE']

### Step 2

In [479]:
#Load the parent dealer view
parent_dealer_view = session.table(PARENT_DEALER_VIEW)

In [480]:
#Checking if parent_dealer_view snowpark dataframe has null values
parent_dealer_view.select(F.sum(F.col("X_DEALER_CODE_HIER").is_null().cast("int")).alias("NUMBER_OF_NULL_VALUES")).show()

---------------------------
|"NUMBER_OF_NULL_VALUES"  |
---------------------------
|30                       |
---------------------------



In [481]:
columns_count = len(parent_dealer_view.columns)
rows_count = parent_dealer_view.count()

print(f"Shape of parent_dealer_view before removing nulls: ({rows_count}, {columns_count})")

Shape of parent_dealer_view before removing nulls: (33172, 26)


In [482]:
#Filtering out null values from the X_DEALER_CODE_HIER column
parent_dealer_view = parent_dealer_view.filter(~F.col("X_DEALER_CODE_HIER").is_null())

columns_count = len(parent_dealer_view.columns)
rows_count = parent_dealer_view.count()

print(f"Shape of parent_dealer_view after removing nulls: ({rows_count}, {columns_count})")

Shape of parent_dealer_view after removing nulls: (33142, 26)


In [483]:
#Extract parent dealer code from the snowpark dataframe
parent_dealer_view = parent_dealer_view.with_column("PARENT_DEALER_CODE",F.trim(F.split(F.col("PAR_ORG_NAME"),F.lit('-'))[0]))

#Select only PARENT_DEALER_CODE and DEALER_CODE
parent_dealer_view = parent_dealer_view.select(F.col("PARENT_DEALER_CODE"),F.col("X_DEALER_CODE_HIER").alias("DEALER_CODE"))

get_shape(parent_dealer_view)

print(f"Before taking distinct : Shape of the dataframe is {get_shape(parent_dealer_view)}")

Before taking distinct : Shape of the dataframe is (33142, 2)


In [484]:
parent_dealer_view = parent_dealer_view.distinct()

print(f"After taking distinct : Shape of the dataframe is {get_shape(parent_dealer_view)}")

After taking distinct : Shape of the dataframe is (11263, 2)


In [485]:
parent_dealer_view.select(F.sum(F.col("PARENT_DEALER_CODE").is_null().cast('int')).alias("NUMBER_OF_NULLS_IN_PARENT_DEALER_CODE")).show()

-------------------------------------------
|"NUMBER_OF_NULLS_IN_PARENT_DEALER_CODE"  |
-------------------------------------------
|0                                        |
-------------------------------------------



### Step 3  - SKU SUPERCEDENCE

In [534]:
#Dropping the UPDATED_ON column
sku_supercedence = (
        session.table(SKU_SUPERCEDENCE_TABLE)
        # .drop('UPDATED_ON')
    )

In [535]:
sku_supercedence = sku_supercedence.with_column('UPDATED_ON',F.to_date(F.col('UPDATED_ON').cast("string"),F.lit('YYYYMMDD')))

In [539]:
latest_update_date_in_sku_supercedence=sku_supercedence.select(F.max("UPDATED_ON")).collect()[0][0]

In [ ]:
sku_supercedence = sku_supercedence.drop("UPDATED_ON")

In [487]:
#Filter for only active SKUs
active_skus = sku_supercedence.filter(F.col('SKUSTATUS') == F.lit('active'))

In [488]:
#Remove " from column names
for old_col in active_skus.columns:
    new_col = old_col.replace('"','')
    active_skus=active_skus.rename(old_col,new_col)

In [489]:
num_active = (
        active_skus
        .group_by('UNIQUE FAMILY CODE')
        .agg(F.count_distinct('SKU').alias('NUM_ACTIVE_SKUS'))
    )

In [490]:
num_active.order_by(F.col("NUM_ACTIVE_SKUS").desc()).show()

--------------------------------------------------------------
|"UNIQUE FAMILY CODE"                    |"NUM_ACTIVE_SKUS"  |
--------------------------------------------------------------
|GLAMOUR<>DISC<>SELF<>CAST<>RED          |6                  |
|SPLENDOR+<>DRUM<>SELF<>CAST<>RED BLACK  |4                  |
|GLAMOUR<>DISC<>SELF<>CAST<>BLUE         |4                  |
|DESTINI<>DRUM<>SELF<>CAST<>WHITE        |4                  |
|XTREME 125<>DISC<>SELF<>CAST<>RED       |4                  |
|SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK      |4                  |
|HF DELUXE<>DRUM<>SELF<>CAST<>BLACK      |4                  |
|GLAMOUR<>DISC<>SELF<>CAST<>BLACK        |4                  |
|XTREME 125<>DRUM<>SELF<>CAST<>RED       |4                  |
|HF DELUXE<>DRUM<>SELF<>CAST<>RED BLACK  |3                  |
--------------------------------------------------------------



In [491]:
num_active.select(F.count_distinct(F.col("UNIQUE FAMILY CODE")).alias("NUMBER_OF_UNIQUE_FAMILY_CODES")).show()

-----------------------------------
|"NUMBER_OF_UNIQUE_FAMILY_CODES"  |
-----------------------------------
|112                              |
-----------------------------------



In [492]:
num_active.select(F.sum(F.col("NUM_ACTIVE_SKUS")).alias("TOTAL_ACTIVE_SKUS")).show()

-----------------------
|"TOTAL_ACTIVE_SKUS"  |
-----------------------
|184                  |
-----------------------



### Step 4

In [493]:
# ── Step 4: ECR — pull all history needed across all forecast months ───────
    # Earliest lookback = 3 months before earliest forecast month (Oct 2025)
    # Latest lookback   = last day before latest forecast month  (Mar 2026)
    # Pulling one wide window; per-month filtering happens in Step 6 via join
# Derive date bounds dynamically from the forecast table
forecast_month_bounds = (
    session.table(TFT_FORECAST_TABLE)
    .select(
        F.min('MONTH_OF_SALE').alias('MIN_FORECAST_MONTH'),
        F.max('MONTH_OF_SALE').alias('MAX_FORECAST_MONTH')
    )
    .collect()[0]
)

earliest_forecast_month = forecast_month_bounds['MIN_FORECAST_MONTH']
latest_forecast_month   = forecast_month_bounds['MAX_FORECAST_MONTH']

# Lookback start = LOOKBACK_MONTHS before the earliest forecast month
lookback_start = F.add_months(F.lit(earliest_forecast_month), -LOOKBACK_MONTHS)

ecr_raw = (
    session.table(ECR_TABLE)
    .filter(F.col('X_CUSTOMER_TYPE').isin(list(CUSTOMER_TYPE)))
    .filter(F.col('CAL_DATE') >= lookback_start)
    .filter(F.col('CAL_DATE') <  F.lit(latest_forecast_month))
    .with_column('CAL_DATE', F.to_date(F.col('CAL_DATE')))
    .with_column(
        'NET_SALES',
        F.col('INVOICED_SALES') + F.col('CANCELLED_SALES') + F.col('RETURNED_SALES')
    )
)

### Step 5

In [494]:
sku_supercedence.select('SKU', 'MODEL', 'UNIQUE FAMILY CODE').show()

--------------------------------------------------------------------------
|"SKU"           |"MODEL"      |"UNIQUE FAMILY CODE"                     |
--------------------------------------------------------------------------
|HDESADRLMCRCBR  |DESTINI 125  |DESTINI<>DRUM<>SELF<>SHEET METAL<>BROWN  |
|HDESADRLMCRPBK  |DESTINI 125  |DESTINI<>DRUM<>SELF<>SHEET METAL<>BLACK  |
|HDESADRLMCRPSW  |DESTINI 125  |DESTINI<>DRUM<>SELF<>SHEET METAL<>WHITE  |
|HDESCDRLMFIBRD  |DESTINI 125  |DESTINI<>DRUM<>SELF<>SHEET METAL<>RED    |
|HDESCDRLMFICBR  |DESTINI 125  |DESTINI<>DRUM<>SELF<>SHEET METAL<>BROWN  |
|HDESCDRLMFIPBK  |DESTINI 125  |DESTINI<>DRUM<>SELF<>SHEET METAL<>BLACK  |
|HDESCDRLMFIPSW  |DESTINI 125  |DESTINI<>DRUM<>SELF<>SHEET METAL<>WHITE  |
|HDESCDRVCFICBR  |DESTINI 125  |DESTINI<>DRUM<>SELF<>CAST<>BROWN         |
|HDESCDRVCFIMRS  |DESTINI 125  |DESTINI<>DRUM<>SELF<>CAST<>SILVER        |
|HDESCDRVCFINRD  |DESTINI 125  |DESTINI<>DRUM<>SELF<>CAST<>RED           |
-------------------------

In [495]:
sku_supercedence=sku_supercedence.rename('UNIQUE FAMILY CODE','UNIQUE_FAMILY_CODE')

In [496]:
# ── Step 5: Map ECR → parent dealer + supercedence ────────────────────────
ecr = (
    ecr_raw
    .join(sku_supercedence.select('SKU', 'MODEL', 'UNIQUE_FAMILY_CODE'),
            on=['SKU', 'MODEL'], how='left')
    .join(parent_dealer_view, on='DEALER_CODE', how='left')
)

# OBD mapping — replace SKU with current OBD variant
if IS_OBD:
    obd = (
        session.table(OBD_MAPPING_TABLE)
        .select(
            F.col('PREVIOUS_OBD_SKU').alias('SKU'),
            F.col('CURRENT_OBD_SKU')
        )
    )
    ecr = (
        ecr
        .join(obd, on='SKU', how='left')
        .with_column(
            'SKU',
            F.coalesce(F.col('CURRENT_OBD_SKU'), F.col('SKU'))
        )
        .drop('CURRENT_OBD_SKU')
    )

In [497]:
# ── Step 6: Cross-join forecast months to ECR, apply rolling 3M filter ────
# This is the Snowpark equivalent of the Python loop over forecast months
forecast_months = forecast.select('MONTH_OF_SALE').distinct()

In [498]:
forecast_months = forecast.select('MONTH_OF_SALE').distinct()

ecr_with_month = ecr.join(forecast_months, how='cross')

ecr_windowed = ecr_with_month.filter(
    (F.col('CAL_DATE') >= F.add_months(F.col('MONTH_OF_SALE'), F.lit(-3))) &
    (F.col('CAL_DATE') <  F.col('MONTH_OF_SALE'))
)

In [499]:
ecr_windowed.group_by(F.col("MONTH_OF_SALE")).agg(F.min("CAL_DATE").alias("MIN_CAL_DATE"),F.max("CAL_DATE").alias("MAX_CAL_DATE")).order_by(F.col("MONTH_OF_SALE").asc()).show()

-----------------------------------------------------
|"MONTH_OF_SALE"  |"MIN_CAL_DATE"  |"MAX_CAL_DATE"  |
-----------------------------------------------------
|2026-01-01       |2025-10-01      |2025-12-31      |
|2026-02-01       |2025-11-01      |2026-01-31      |
|2026-03-01       |2025-12-01      |2026-02-28      |
|2026-04-01       |2026-01-01      |2026-03-31      |
-----------------------------------------------------



In [500]:
ecr_windowed.select(F.sum(F.col("PARENT_DEALER_CODE").is_null().cast('int')).alias("NUMBER_OF_NULLS_IN_PARENT_DEALER_CODE")).show()

-------------------------------------------
|"NUMBER_OF_NULLS_IN_PARENT_DEALER_CODE"  |
-------------------------------------------
|0                                        |
-------------------------------------------



In [501]:
 # ── Step 7: SKU-level sales per dealer-family per forecast month ──────────

In [502]:
dealer_family_sku_sales = (
        ecr_windowed
        .group_by(['MONTH_OF_SALE', 'PARENT_DEALER_CODE', 'UNIQUE_FAMILY_CODE', 'SKU'])
        .agg(F.sum('NET_SALES').alias('DEALER_SKU_SALES'))
    )

In [503]:
# # Family-level totals (sum of active SKU sales = denominator for weights)
# dealer_family_sales = (
#         dealer_family_sku_sales
#         .group_by(['MONTH_OF_SALE', 'PARENT_DEALER_CODE', 'UNIQUE_FAMILY_CODE'])
#         .agg(F.sum('DEALER_SKU_SALES').alias('FAMILY_TOTAL_SALES'))
#     )

In [504]:
#top 5 rows of dealer_family_sku_sales snowpark dataframe
top_5_rows_dealer_family_sku_sales = dealer_family_sku_sales.limit(10).to_pandas()

top_5_rows_dealer_family_sku_sales.head()

,MONTH_OF_SALE,PARENT_DEALER_CODE,UNIQUE_FAMILY_CODE,SKU,DEALER_SKU_SALES
0,2026-04-01,11840,SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK PURPLE,HSPPCDRSCFIBKP,0.0
1,2026-03-01,10282,None,EC27HSPSADSSCFIBLK,0.0
2,2026-02-01,10670,PLEASURE+<>DRUM<>SELF<>CAST<>YELLOW,HPLNSHRZCFIJYL,0.0
3,2026-02-01,10980,XTREME 160<>DISC<>SELF<>CAST<>GREY,HXTRGDDSCFIMAG,0.0
4,2026-02-01,11047,SPLENDOR+<>DRUM<>SELF<>CAST<>RED BLACK,HSPPCDRSCFIRBK,0.0


### Step 8

In [505]:
#top 10 rows of forecast snowpark dataframe
top_10_rows_forecast = forecast.limit(10).to_pandas()
top_10_rows_forecast.head()

,PARENT_DEALER_CODE_MODEL_FAMILY,PREDICTED_SALES_TFT,MONTH_OF_SALE,PARENT_DEALER_CODE,UNIQUE_FAMILY_CODE
0,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,3.27,2026-02-01,12015,PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY GREY
1,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,3.46,2026-01-01,12015,PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY GREY
2,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,3.29,2026-03-01,12015,PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY GREY
3,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,2.93,2026-04-01,12015,PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY GREY
4,12047<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,5.31,2026-02-01,12047,PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY GREY


In [506]:
def removing_double_quotes_from_column_names(df):
    for old_col in df.columns:
        new_col = old_col.replace('"', '')
        df = df.with_column_renamed(old_col, new_col) 
    return df 

num_active = removing_double_quotes_from_column_names(num_active)
active_skus = removing_double_quotes_from_column_names(active_skus)


In [507]:
active_skus = active_skus.with_column_renamed('UNIQUE FAMILY CODE','UNIQUE_FAMILY_CODE')

In [508]:
num_active = num_active.with_column_renamed('UNIQUE FAMILY CODE','UNIQUE_FAMILY_CODE')

In [509]:
# ── Step 8: Expand forecast to SKU level via active SKU supercedence ──────
forecast_sku = (
        forecast
        .join(
            active_skus.select(F.col('UNIQUE_FAMILY_CODE'), 'SKU'),
            on='UNIQUE_FAMILY_CODE',
            how='left'
        )
        .join(
            num_active,
            on='UNIQUE_FAMILY_CODE', 
            how='left'
        )
    )

In [510]:
forecast_sku.select(F.sum(F.col('SKU').is_null().cast('int')).alias("NUMBER_OF_NULL_VALUES_IN_SKU_COLUMN")).show()

-----------------------------------------
|"NUMBER_OF_NULL_VALUES_IN_SKU_COLUMN"  |
-----------------------------------------
|15149                                  |
-----------------------------------------



In [511]:
dealer_family_sku_sales.columns

['MONTH_OF_SALE',
 'PARENT_DEALER_CODE',
 'UNIQUE_FAMILY_CODE',
 'SKU',
 'DEALER_SKU_SALES']

In [512]:
forecast_sku.columns

['UNIQUE_FAMILY_CODE',
 'PARENT_DEALER_CODE_MODEL_FAMILY',
 'PREDICTED_SALES_TFT',
 'MONTH_OF_SALE',
 'PARENT_DEALER_CODE',
 'SKU',
 'NUM_ACTIVE_SKUS']

### Step 9

In [513]:
# # ── Step 9: Join SKU weights onto expanded forecast ───────────────────────
# disaggregated = (
#     forecast_sku
#     # Stage 1: Join SKU-specific sales history (using all 4 keys)
#     .join(
#         dealer_family_sku_sales.select(
#             'MONTH_OF_SALE', 'PARENT_DEALER_CODE', 'UNIQUE_FAMILY_CODE',
#             'SKU', 'DEALER_SKU_SALES'
#         ),
#         on=['MONTH_OF_SALE', 'PARENT_DEALER_CODE', 'UNIQUE_FAMILY_CODE', 'SKU'],
#         how='left'
#     )
#     # Stage 2: Join Family-wide totals (using only the 3 family keys!)
#     .join(
#         dealer_family_sales,
#         on=['MONTH_OF_SALE', 'PARENT_DEALER_CODE', 'UNIQUE_FAMILY_CODE'],
#         how='left'
#     )
#     .with_column('DEALER_SKU_SALES', F.coalesce(F.col('DEALER_SKU_SALES'), F.lit(0.0)))
#     .with_column('FAMILY_TOTAL_SALES', F.coalesce(F.col('FAMILY_TOTAL_SALES'), F.lit(0.0)))
# )

disaggregated = (
    forecast_sku
    .join(
        dealer_family_sku_sales.select(
            'MONTH_OF_SALE', 'PARENT_DEALER_CODE', 'UNIQUE_FAMILY_CODE',
            'SKU', 'DEALER_SKU_SALES'
        ),
        on=['MONTH_OF_SALE', 'PARENT_DEALER_CODE', 'UNIQUE_FAMILY_CODE', 'SKU'],
        how='left'
    )
    .with_column('DEALER_SKU_SALES', F.coalesce(F.col('DEALER_SKU_SALES'), F.lit(0.0)))
)

# NEW STEP: Calculate FAMILY_TOTAL_SALES here so it only sums ACTIVE SKUs!
family_window = Window.partition_by('MONTH_OF_SALE', 'PARENT_DEALER_CODE', 'UNIQUE_FAMILY_CODE')

disaggregated = disaggregated.with_column(
    'FAMILY_TOTAL_SALES',
    F.sum('DEALER_SKU_SALES').over(family_window)
)

In [514]:
# forecast.columns

In [515]:
# disaggregated.columns

In [516]:
# ── Step 10: Compute SKU weight (mirrors percentsku logic exactly) ─────────
disaggregated = disaggregated.with_column(
        'PERCENT_PROPORTION',
        F.when(
            # No history at all for this family → equal split
            F.col('FAMILY_TOTAL_SALES') == F.lit(0),
            F.lit(1.0) / F.col('NUM_ACTIVE_SKUS')
        ).when(
            # Family has history but this SKU has zero → gets 0
            F.col('DEALER_SKU_SALES') == F.lit(0),
            F.lit(0.0)
        ).otherwise(
            F.col('DEALER_SKU_SALES') / F.col('FAMILY_TOTAL_SALES')
        )
    )

### Step 11

In [517]:
# ── Step 11: Disaggregate predicted sales ─────────────────────────────────
disaggregated = disaggregated.with_column(
    'PREDICTED_SALES_SKU_TFT',
    F.when(
        F.col('NUM_ACTIVE_SKUS') == F.lit(1),
        F.col('PREDICTED_SALES_TFT')                              # single SKU — no split needed
    ).otherwise(
        F.col('PERCENT_PROPORTION') * F.col('PREDICTED_SALES_TFT')
    )
)

### Step 12

In [518]:
# ── Step 12: Flag families with no ECR history ────────────────────────────
disaggregated = disaggregated.with_column(
    'NO_HISTORY_FLAG',
    F.when(F.col('FAMILY_TOTAL_SALES') == F.lit(0), F.lit(True))
        .otherwise(F.lit(False))
)

### Step 13

In [519]:
# ── Step 13: Final output columns ─────────────────────────────────────────
output = disaggregated.select(
    'MONTH_OF_SALE',
    'PARENT_DEALER_CODE_MODEL_FAMILY',
    'PARENT_DEALER_CODE',
    'UNIQUE_FAMILY_CODE',
    'SKU',
    'NUM_ACTIVE_SKUS',
    'PREDICTED_SALES_TFT',
    F.round('PERCENT_PROPORTION', 5).alias('PERCENT_PROPORTION'),
    F.round('PREDICTED_SALES_SKU_TFT', 4).alias('PREDICTED_SALES_SKU_TFT'),
    'DEALER_SKU_SALES',
    'FAMILY_TOTAL_SALES',
    'NO_HISTORY_FLAG'
)

### Step 14

In [520]:
# ── Step 14: Sanity check before writing ──────────────────────────────────
check = (
    output.filter(F.col('NO_HISTORY_FLAG') == F.lit(False))
    .group_by(['MONTH_OF_SALE', 'PARENT_DEALER_CODE_MODEL_FAMILY'])
    .agg(
        F.max('PREDICTED_SALES_TFT').alias('ORIGINAL'),
        F.sum('PREDICTED_SALES_SKU_TFT').alias('REAGGREGATED')
    )
    .with_column('DIFF', F.abs(F.col('ORIGINAL') - F.col('REAGGREGATED')))
)

max_diff         = check.agg(F.max('DIFF').alias('MAX_DIFF')).collect()[0]['MAX_DIFF']
no_history_count = output.filter(F.col('NO_HISTORY_FLAG') == F.lit(True)).count()
total_rows       = output.count()

print(f"Total output rows        : {total_rows}")
print(f"NO_HISTORY_FLAG rows     : {no_history_count}")
print(f"Max reaggregation diff   : {max_diff:.6f}")

Total output rows        : 180971
NO_HISTORY_FLAG rows     : 21451
Max reaggregation diff   : 0.000200
